In [0]:
# Reads portfolio configs from Backstop/config/live/*.json and outputs them
# as a JSON array for use by a for_each_task downstream.

dbutils.widgets.text("portfolio", "")  # comma-separated filter, empty = all
dbutils.widgets.text("config_folder", "live")

import os
import sys
import json

try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    repo_root = "/Workspace" + notebook_path.rsplit("/DailyLoan/", 1)[0]
    sys.path.insert(0, os.path.join(repo_root, "aws_utils"))
    sys.path.insert(0, os.path.join(repo_root, "01-utils"))
except Exception:
    sys.path.append("../../aws_utils")
    sys.path.append("../../01-utils")

from helpers import load_configs

In [0]:
config_folder = dbutils.widgets.get("config_folder").strip() or "live"
portfolio_raw = dbutils.widgets.get("portfolio").strip()
filter_set = {p.strip().lower() for p in portfolio_raw.split(",") if p.strip()} if portfolio_raw else None

In [0]:
configs = load_configs(dbutils, config_folder, filter_set, report_name="DailyLoan")
print(f"Loaded {len(configs)} portfolio config(s) from '{config_folder}'")

# Build the iteration list
iterations = []
for config in configs:
    report_args = config["report_args"].copy()
    portfolio_name = report_args["portfolio_name"]
    iterations.append({
        "report_args": json.dumps(report_args),
        "report_name": f"{portfolio_name}_DailyLoan"
    })

print(f"Portfolios to run: {[item['report_name'] for item in iterations]}")

In [0]:
# Output as task value for for_each_task
# dbutils.jobs.taskValues.set(key="portfolio_configs", value=json.dumps(iterations))
dbutils.jobs.taskValues.set(key="portfolio_configs", value=iterations)
dbutils.notebook.exit(json.dumps(iterations))